## Imports

In [1]:
import argparse
from tqdm import tqdm
import os
from os.path import join, dirname
import gc
import sys
import time
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from IPython.display import HTML

import torch
import torch.nn.functional as F
from torch.nn.parallel.data_parallel import DataParallel

import torchvision
import torchaudio
from pytorchvideo.data.encoded_video import EncodedVideo
import torchshow as ts
from einops import rearrange, repeat

sns.set()

%env XRT_TPU_CONFIG=localservice;0;localhost:51011
%env PT_XLA_DEBUG=1
%env USE_TORCH=ON
%env XLA_USE_BF16=1
# %env XLA_IR_DEBUG=1 
# %env XLA_HLO_DEBUG=1
%load_ext autoreload
%autoreload 2

env: XRT_TPU_CONFIG=localservice;0;localhost:51011
env: PT_XLA_DEBUG=1
env: USE_TORCH=ON
env: XLA_USE_BF16=1


In [2]:
sys.path.append(join(dirname(dirname(os.getcwd()))))

In [11]:
sys.path.append(join(dirname(dirname(os.getcwd()))))

from hand4whole.main.model import get_model
from hand4whole.common.utils.preprocessing import process_bbox, generate_patch_image
from hand4whole.common.utils.human_models import smpl_x
from hand4whole.common.utils.vis import render_mesh, save_obj

from configuration import cf
from run_setup import _mp_fn, FLAGS
from memory_perceiver.datasets.stream_data import Streamer
from memory_perceiver.datasets.my_sampler import MyDistributedBatchSampler
from memory_perceiver.processing.box_detection import interpolate_bbox, Resize_with_pad
from memory_perceiver.processing.vectorize import input_to_tensor, fourier_encode
from memory_perceiver.rendering.renderer import render_silhouettes, batch_intrinsic_camera, video_masking, write_output
from memory_perceiver.core.model_net import PerceiverIO
from memory_perceiver.core.miscellaneous import batch_dictionary, batch_dictionary_to_list

ModuleNotFoundError: No module named 'nets'

In [ ]:
if __name__ == '__main__':
    xmp.spawn(_mp_fn, args=(FLAGS,), nprocs=FLAGS.num_cores)

In [5]:
deviceEval = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device=xm.xla_device()
print(xm.get_xla_supported_devices())

t1 = torch.randn(3,3,device=device)
t2 = torch.randn(3,3,device=device)
print(t1 + t2)

['xla:1', 'xla:2', 'xla:3', 'xla:4', 'xla:5', 'xla:6', 'xla:7', 'xla:8']
tensor([[-1.1875, -0.7148, -0.4160],
        [-0.3262, -0.5273, -0.8828],
        [-0.8555, -0.5820,  0.3262]], device='xla:1')


2022-12-30 10:58:30.706228: E tensorflow/core/framework/op_kernel.cc:1676] OpKernel ('op: "TPURoundRobin" device_type: "CPU"') for unknown op: TPURoundRobin
2022-12-30 10:58:30.706285: E tensorflow/core/framework/op_kernel.cc:1676] OpKernel ('op: "TpuHandleToProtoKey" device_type: "CPU"') for unknown op: TpuHandleToProtoKey


In [6]:
# smpl-x snapshot load

model_path = '../../hand4whole/demo/snapshot_6.pth.tar'
assert os.path.exists(model_path), 'Cannot find model at ' + model_path
print('Load checkpoint from {}'.format(model_path))
model_smplx = get_model('test')
model_smplx = DataParallel(model_smplx)
ckpt = torch.load(model_path, map_location=torch.device("cpu"))
model_smplx.load_state_dict(ckpt['network'], strict=False)
model_smplx = model_smplx.eval().to(device)

Load checkpoint from ../../hand4whole/demo/snapshot_6.pth.tar


# Training

In [7]:
# view data

# dataset = Streamer(join(os.path.dirname(os.getcwd()),'data_test'), 1, 0.25, 3, clean_audio=True)
# def my_collate(batch):
#     path_ids = [item[0] for item in batch]
#     stream_nbs = [item[1] for item in batch]
#     inputs = [item[2] for item in batch]
#     targets = [item[3] for item in batch]
#     return (path_ids, stream_nbs, inputs, targets)
# train_loader = torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=False, num_workers=0, collate_fn=my_collate, drop_last=True, pin_memory=True, sampler=None)
# iter_batch = iter(train_loader)
# path_id, stream_nb, input, target = next(iter_batch)
# print(path_id, stream_nb)

# input[0] = { key : torch.tensor(value) for key, value in input[0].items()}
# rendered_video = video_masking(input[0]["video"], input[0], model_smplx)
# ani=ts.show_video(rearrange(rendered_video, 'c f h w -> f c h w'))
# HTML(ani.to_jshtml())

In [8]:
# Model parameters

nb_frames = 32
frames_future = 8
len_video_frame = 1824  # 456*256/(8*8)
len_audio_frame = 120   # 1920/16
query_dim = 832
video_latent_dim = 256
audio_latent_dim = 560
num_workers = 2
batch_size = 1

len_vid = nb_frames*len_video_frame
len_audio = nb_frames*len_audio_frame
len_audio_raw = len_audio*cf.sample_per_frame
video_future = frames_future*len_video_frame
audio_future = frames_future*len_audio_frame

special_compression = [i for i in range(video_future)] + [i for i in range(len_vid, len_vid+audio_future)]
for k in range(len(cf.body_parts)):
    special_compression += [ len_vid + len_audio + k*nb_frames + i for i in range(frames_future) ]

In [9]:
def create_model(device=device):

    # Projection to return modalities, enhancement

    modality_parameters = {}
    modality_parameters['audio'] = audio_latent_dim
    modality_parameters['video'] = video_latent_dim
    for part, params in cf.body_parts.items():
        modality_parameters[part] = query_dim - 2*128 - params

    projection_parameters = dict(cf.body_parts)
    projection_parameters.update({'audio': 16})

    # Instantiate model
        
    model = PerceiverIO(
            2,
            query_dim,
            special_compression = special_compression,
            device=device,
            projection_parameters = projection_parameters,
            modality_parameters = modality_parameters,
            max_batch_size = 1
        )
    model = model.to(device)

    # Queries

    queries = []
    audio_query = repeat( torch.cat([model.modality['audio'], torch.zeros(16, device=device)]), '... -> n ...', n=audio_future)
    audio_pos = fourier_encode(torch.linspace(-1, 1, audio_future), audio_future/2, num_bands=128).to(device)
    queries.append(torch.cat([audio_query, audio_pos], dim=-1).to(device))
    for part, params in cf.body_parts.items():
        part_query = repeat( torch.cat([model.modality[part], torch.zeros(params, device=device)]), '... -> n ...', n=frames_future)
        part_pos = fourier_encode(torch.linspace(-1, 1, frames_future), frames_future/2, num_bands=128).to(device)
        queries.append(torch.cat([part_query, part_pos], dim=-1).to(device))
    queries=torch.cat(queries)


    return model, queries

In [10]:
# Dataloader

learning_rate = 0.001
num_epochs =  100
acting_ratio = 0.5
batch_size=1
duration_total=2
duration = 1
duration_future = 0.25
max_stream_nbs = (duration_total-duration)/duration_future
dataset = Streamer(join(os.path.dirname(os.getcwd()),'data_test'), duration, duration_future, duration_total,batch_size, clean_audio=True)

sampler = None
if (xm.xrt_world_size() > 1):
    sampler = MyDistributedBatchSampler( #torch.utils.data.distributed.DistributedSampler
            dataset,
            num_replicas=xm.xrt_world_size(),
            rank=xm.get_ordinal(),
            shuffle=False)

def my_collate(batch):
    path_ids = [item[0] for item in batch]
    stream_nbs = [item[1] for item in batch]
    inputs = [item[2] for item in batch]
    targets = [item[3] for item in batch]
    return (path_ids, stream_nbs, inputs, targets)
train_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=32, collate_fn=my_collate, drop_last=True, pin_memory=True, sampler=sampler)

In [11]:
i, (path_ids, stream_nbs, inputs, targets) = next(enumerate(train_loader))

In [12]:
earear

NameError: name 'earear' is not defined

In [ ]:
# Autoregressive mode

model, queries = create_model()
model = model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
train_loss = []

for epoch in tqdm(range(num_epochs)):
    target_display = {part: torch.tensor([], device=device) for part in ['video', 'audio']+list(cf.body_parts.keys())}
    pred_display = {part: torch.tensor([], device=device) for part in ['audio']+list(cf.body_parts.keys())}
    last_path_id = None
    pose_shapes = None
    len_loader = int(len(train_loader)/batch_size)
    for i, (path_ids, stream_nbs, inputs, targets) in tqdm(enumerate(train_loader), total=len_loader):

        print(path_ids, stream_nbs)

        # Clear memory for new sequence
        
        if( path_ids[0] != last_path_id):
            last_path_id = path_ids[0]
            model.reset_memories()
            audio_mem = torch.tensor([], device=device)
            body_mem = {part: torch.tensor([], device=device) for part in list(cf.body_parts.keys())}
        pose_shapes = [torch.mean(torch.tensor(input['smplx_shape']), dim=0) for input in inputs]
        acting =  (stream_nbs[0]+1)/max_stream_nbs >= acting_ratio

        print('acting: ', acting)
        t0 = time.time()
        
        # append previous predictions and mask

        if( len(audio_mem) > 0  and acting):
            _, len_audio_mem = audio_mem.shape
            _, len_body_mem, _ = list(body_mem.values())[0].shape
            for i in range(len(inputs)):
                audio_mem_filled = torch.cat([inputs[i]["audio"][:-len_audio_mem].to(device), audio_mem[i]])
                inputs[i]["audio"] =  ( inputs[i]["audio"] + audio_mem_filled )/2
                for part in cf.body_parts.keys():
                    torch.cat([torch.tensor(inputs[i][part], device=device)[:-len_body_mem, :], body_mem[part][i]])
                inputs_i_vect = {part: item.to(device) if(torch.is_tensor(item)) else torch.tensor(item).to(device) for part, item in inputs[i].items()}
                inputs[i]["video"] = video_masking(inputs[i]["video"]*255, inputs_i_vect, model_smplx)/255
            
        print('concatenate memory and mask: ', time.time()-t0)
        t0 = time.time()

        # remember target 

        if(epoch % 10 == 0 and path_ids[0] == 0 ):
            target_display['video'] = torch.cat([target_display['video'], targets[0]['video']], dim=1)
            target_display['audio'] = torch.cat([target_display['audio'], targets[0]['audio']], dim=0)
            for part in cf.body_parts.keys():
                target_display[part] = torch.cat([target_display[part], torch.tensor(targets[0][part], device=device)], dim=0)
                    
        # vectorize

        targets_audio = torch.tensor([target['clean_audio'] for target in targets]).to(device)
        [ target.pop(key) for key in ['video','audio','clean_audio','smplx_shape'] for target in targets ]
        targets_body  = batch_dictionary(targets)
        targets_body = {part: torch.tensor(item).to(device) for part, item in targets_body.items()}
        [ input.pop(key) for key in ['clean_audio','smplx_shape'] for input in inputs ]
        inputs = batch_dictionary(inputs)
        inputs['video'] = torch.stack(inputs['video'])
        inputs['audio'] = torch.stack(inputs['audio'])
        inputs = {part: item.to(device) if(torch.is_tensor(item)) else torch.tensor(item).to(device) for part, item in inputs.items()}
        batch = input_to_tensor(inputs, model.modality, cf.body_parts).to(device)

        print('vectorize: ', time.time()-t0)
        t0 = time.time()

        # zero the parameter gradients

        optimizer.zero_grad()

        # forward + backprop + step

        x, aux_loss = model(batch, queries = queries)
        loss = aux_loss
        output = {}
        print(acting)
        if(acting):
            loss_flow = 1
        else:
            loss_flow = 0
        output['audio'] = rearrange(model.projections['audio'](x[:, :audio_future, :]) ,  "b t t1 -> b (t t1)")
        start_modality = audio_future
        loss = loss + loss_flow*F.l1_loss(output['audio'], targets_audio)
        for part, params in cf.body_parts.items():
            output[part] = model.projections[part](x[:, start_modality:start_modality+frames_future, :])
            start_modality += frames_future
            loss = loss + loss_flow*F.l1_loss(output[part], targets_body[part])
        loss.backward()
        optimizer.step()
        xm.mark_step()

        print('train: ', time.time()-t0)
        t0 = time.time()

        # remember pred 

        if(epoch % 10 == 0 and path_ids[0] == 0 ):
            for part in pred_display.keys():
                pred_display[part] = torch.cat([pred_display[part], torch.tensor(output[part][0])], dim=0)


        # update params and memory for autoregression

        if(acting):
            audio_mem = torch.cat([audio_mem, output['audio']], dim=1)
            audio_mem = audio_mem[:, max(-len(audio_mem), -len_audio_raw):].clone().detach()
            for part in cf.body_parts.keys():
                body_mem[part] = torch.cat([body_mem[part], output[part]], dim=1)
                body_mem[part] = body_mem[part][:, max(-len(body_mem[part]), -nb_frames):, :].clone().detach()

        print(loss.detach().item())

        print('Append to memory: ', time.time()-t0)

        # reset memory

        train_loss.append(loss.detach().item())

    # display

    if( epoch % 10 == 0 ):
        print(torch_xla.debug.metrics.metrics_report())
        plt.plot(train_loss)
        plt.show()

    # Plot predictions

    if(epoch % 10 == 0 ):
        write_output(target_display['video'], target_display['audio'], target_display, pose_shapes[0], epoch, device, model_smplx, 'target')
        write_output(target_display['video'], pred_display['audio'], pred_display, pose_shapes[0], epoch, device, model_smplx, 'pred')


In [ ]:
print(torch_xla._XLAC._xla_tensors_report(0, str(device)))

In [ ]:
# stats

# torch.cuda.memory_stats(device)
# torch.cuda.empty_cache()

objects = []
mem_total = 0
for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) or (hasattr(obj, 'data') and torch.is_tensor(obj.data)):
            mem = (obj.element_size() * obj.nelement()/1000000)
            mem_total += mem
            objects.append( [type(obj), obj.size(), obj.nelement(), mem])
    except:
        pass
print(mem_total)
objects = sorted(objects, key = lambda x: x[2], reverse=True)
for obj in objects:
    print(obj)